In [169]:
# Instalando dependencias para lidar com arquivos parquet.
!pip install -q pyarrow fastparquet

In [170]:
import pandas as pd
import pyarrow, fastparquet

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [171]:
# Caminho base para os arquivos da camada gold
BASE_GOLD   = "/content/drive/MyDrive/projeto-medalhao/gold/"

In [172]:
# Método para incrementar a tabela resultante da camada gold 'ml_aluno' com dados socioeconomicos,
# buscando caracterizar melhor municípios e UFs brasileiras.
def constroi_df_resultante(df):
  df_estatisticas_escolares = pd.read_parquet(f'{BASE_GOLD}/estatisticas_escolares', engine='pyarrow')
  df_dados_socioeconomicos = pd.read_csv("/content/br_dados_socioeconomicos.csv", delimiter=";")

  df_estatisticas_escolares = df_estatisticas_escolares \
  .filter(items=["ano", "id_escola", "total_alunos", "percentual_faltantes", "desvio_padrao_proficiencia"]) \
  .rename(columns={
      "total_alunos": "total_participantes_escola",
      "percentual_faltantes": "percentual_faltantes_escola",
      "desvio_padrao_proficiencia": "desvio_padrao_proficiencia_escola"})

  df = df.merge(df_estatisticas_escolares, on=["id_escola", "ano"], how="left")

  df['nome_municipio'] = df['nome_municipio'].str.upper()
  df_dados_socioeconomicos = df_dados_socioeconomicos.rename(columns={"municipio": "nome_municipio"})
  df = df.merge(df_dados_socioeconomicos, on=["nome_municipio", "sigla_uf", "ano"], how="left")

  return df.drop(columns=["_gold_processed_at"])

In [173]:
df_ml_aluno = pd.read_parquet(f'{BASE_GOLD}/ml_aluno', engine='pyarrow')
df_uf = pd.read_parquet(f'{BASE_GOLD}/consolidado_uf', engine='pyarrow')

# Encontramos alguns valores faltantes para média de português municipal e estadual, que não haviam sido percebidos durante a construção do pipeline de dados.

###**Estadual**:
    Foram encontrados valores faltantes para sigla_uf = ("Não encontrado", "DF", "TO")
      
        Não encontrado (489 valores): não houve match entre o id do município reportado para aquela escola e o mapeamento de ids municipais fornecido.
        Decisão - descartar.
        
        DF (22111 valores): ao consultar a tabela de dados brutos estaduais do Tech Challenge 2, percebemos que ela não continha dados para o Distrito Federal.
        Decisão - utilizar a média do único município que compõe o estado (Brasília).

        TO (24 valores): uma investigação mais detalhada no pipeline é necessária para entender o motivo pelo qual registros da rede privada da cidade de Gurupi, em 2024, ficaram com valores nulos para média municipal e estadual.
        Decisão - popular com o valor conhecido de média estadual para TO em 2024 (742.86).

In [174]:
df_ml_aluno.describe()

,id_aluno,id_escola,feat_rede_encoded,feat_peso_aluno,feat_media_proficiencia_escola,feat_media_portugues_municipio,feat_media_portugues_estado,target_alfabetizado,target_nivel_alfabetizacao,_gold_processed_at
count,3.354661e+06,3.354661e+06,3.354661e+06,3.354661e+06,3.354661e+06,3.354456e+06,3.332037e+06,3.354661e+06,3.354661e+06,3354661
mean,3.209517e+07,6.002180e+07,1.111090e+00,1.148525e+00,7.483799e+02,7.480113e+02,7.477786e+02,5.915787e-01,4.428456e+00,2026-08-26 02:22:27.232228608
min,1.100000e+07,6.000000e+07,1.000000e+00,1.000000e-01,6.010600e+02,6.733000e+02,7.125600e+02,0.000000e+00,0.000000e+00,2026-08-26 02:22:27.232231
25%,2.502555e+07,6.001124e+07,1.000000e+00,1.000000e+00,7.328700e+02,7.358000e+02,7.373000e+02,0.000000e+00,3.000000e+00,2026-08-26 02:22:27.232230912
50%,3.118988e+07,6.002210e+07,1.000000e+00,1.090000e+00,7.469500e+02,7.461800e+02,7.472800e+02,1.000000e+00,5.000000e+00,2026-08-26 02:22:27.232230912
75%,4.104093e+07,6.003280e+07,1.000000e+00,1.210000e+00,7.614900e+02,7.568300e+02,7.546800e+02,1.000000e+00,6.000000e+00,2026-08-26 02:22:27.232230912
max,5.302770e+07,6.004281e+07,4.000000e+00,1.425500e+02,8.850200e+02,8.684600e+02,7.973400e+02,1.000000e+00,8.000000e+00,2026-08-26 02:22:27.232231
std,1.033263e+07,1.254820e+04,3.143114e-01,3.585184e-01,2.465367e+01,2.030885e+01,1.556682e+01,4.915419e-01,1.877076e+00,NaN


In [175]:
# Olhando para a media estadual.
df_ml_aluno[df_ml_aluno['feat_media_portugues_estado'].isna() == True]["sigla_uf"].count()

np.int64(22624)

In [176]:
df_ml_aluno[df_ml_aluno['feat_media_portugues_estado'].isna() == True]["sigla_uf"].unique()

array(['Não encontrado', 'DF', 'TO'], dtype=object)

In [177]:
# Descartando não encontrados.
df_ml_aluno = df_ml_aluno[df_ml_aluno['sigla_uf'] != "Não encontrado"]

# Atribuindo o valor conhecido de média estadual para TO em 2024.
MEDIA_TO_2024 = 742.86
df_ml_aluno.loc[df_ml_aluno['feat_media_portugues_estado'].isna() & (df_ml_aluno['sigla_uf'] == 'TO'), 'feat_media_portugues_estado'] = MEDIA_TO_2024

# Atribuindo a média ao Distrito Federal.
MEDIA_BRASILIA_2024 = 743.01
df_ml_aluno.loc[(df_ml_aluno['sigla_uf'] == 'DF') & (df_ml_aluno['ano'] == 2024), 'feat_media_portugues_estado'] = MEDIA_BRASILIA_2024

In [178]:
df_ml_aluno.describe()

,id_aluno,id_escola,feat_rede_encoded,feat_peso_aluno,feat_media_proficiencia_escola,feat_media_portugues_municipio,feat_media_portugues_estado,target_alfabetizado,target_nivel_alfabetizacao,_gold_processed_at
count,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.354172e+06,3.353967e+06,3.354172e+06,3.354172e+06,3.354172e+06,3354172
mean,3.209486e+07,6.002180e+07,1.111085e+00,1.148532e+00,7.483780e+02,7.480095e+02,7.477471e+02,5.915684e-01,4.428381e+00,2026-08-26 02:22:27.232228864
min,1.100000e+07,6.000000e+07,1.000000e+00,1.000000e-01,6.010600e+02,6.733000e+02,7.125600e+02,0.000000e+00,0.000000e+00,2026-08-26 02:22:27.232231
25%,2.502554e+07,6.001124e+07,1.000000e+00,1.000000e+00,7.328700e+02,7.358000e+02,7.373000e+02,0.000000e+00,3.000000e+00,2026-08-26 02:22:27.232230912
50%,3.118988e+07,6.002210e+07,1.000000e+00,1.090000e+00,7.469500e+02,7.461800e+02,7.472800e+02,1.000000e+00,5.000000e+00,2026-08-26 02:22:27.232230912
75%,4.104087e+07,6.003279e+07,1.000000e+00,1.210000e+00,7.614900e+02,7.568300e+02,7.546800e+02,1.000000e+00,6.000000e+00,2026-08-26 02:22:27.232230912
max,5.302770e+07,6.004281e+07,4.000000e+00,1.425500e+02,8.850200e+02,8.684600e+02,7.973400e+02,1.000000e+00,8.000000e+00,2026-08-26 02:22:27.232231
std,1.033286e+07,1.254817e+04,3.143056e-01,3.585416e-01,2.465181e+01,2.030606e+01,1.552018e+01,4.915438e-01,1.877040e+00,NaN


In [179]:
df_ml_aluno[df_ml_aluno['feat_media_portugues_estado'].isna() == True]["sigla_uf"].count()

np.int64(0)

###**Municipal**
    Para os municípios, a estratégia foi atribuir aos registros faltantes o valor da média das outras redes de ensino naquele mesmo ano.
    Quando não disponível, foi atribuída a média estadual.

In [180]:
# Olhando para a media municipal.
mun_medias_faltantes = df_ml_aluno[df_ml_aluno['feat_media_portugues_municipio'].isna() == True]["nome_municipio"].unique().tolist()

In [182]:
df_ml_aluno[df_ml_aluno['feat_media_portugues_municipio'].isna() == True].head(220)

,id_aluno,id_escola,nome_municipio,sigla_uf,feat_rede_encoded,feat_peso_aluno,feat_media_proficiencia_escola,feat_media_portugues_municipio,feat_media_portugues_estado,target_alfabetizado,target_nivel_alfabetizacao,_gold_processed_at,ano
62385,16000472,60003527,Ferreira Gomes,AP,2,1.08,689.94,NaN,731.43,0,3,2026-08-26 02:22:27.232231,2023
62386,16000462,60003527,Ferreira Gomes,AP,2,1.08,689.94,NaN,731.43,0,2,2026-08-26 02:22:27.232231,2023
62387,16000473,60003527,Ferreira Gomes,AP,2,1.08,689.94,NaN,731.43,0,2,2026-08-26 02:22:27.232231,2023
62388,16000464,60003527,Ferreira Gomes,AP,2,1.08,689.94,NaN,731.43,0,3,2026-08-26 02:22:27.232231,2023
62389,16000475,60003527,Ferreira Gomes,AP,2,1.08,689.94,NaN,731.43,0,2,2026-08-26 02:22:27.232231,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3314485,43027613,60038812,Charrua,RS,1,1.71,704.63,NaN,733.93,0,4,2026-08-26 02:22:27.232231,2024
3314486,43027606,60038812,Charrua,RS,1,1.71,704.63,NaN,733.93,0,4,2026-08-26 02:22:27.232231,2024
3314487,43027610,60038812,Charrua,RS,1,1.71,704.63,NaN,733.93,0,3,2026-08-26 02:22:27.232231,2024
3314488,43027608,60038812,Charrua,RS,1,1.71,704.63,NaN,733.93,0,4,2026-08-26 02:22:27.232231,2024


In [183]:
for municipio in mun_medias_faltantes:
  for ano in [2023, 2024]:

    total_municipais_nulas = df_ml_aluno.loc[(df_ml_aluno['nome_municipio'] == municipio) & (df_ml_aluno['ano'] == ano), 'feat_media_portugues_municipio'].isna().sum()

    if total_municipais_nulas == \
     df_ml_aluno.loc[(df_ml_aluno['nome_municipio'] == municipio) & (df_ml_aluno['ano'] == ano), 'feat_media_portugues_municipio'].isna().count():

      df_ml_aluno.loc[(df_ml_aluno['nome_municipio'] == municipio) & (df_ml_aluno['ano'] == ano), 'feat_media_portugues_municipio'] = \
        df_ml_aluno.loc[(df_ml_aluno['nome_municipio'] == municipio) & (df_ml_aluno['ano'] == ano), 'feat_media_portugues_estado'].unique().mean()

    elif total_municipais_nulas > 0:

      df_ml_aluno.loc[
          (df_ml_aluno['feat_media_portugues_municipio'].isna() == True) & (df_ml_aluno['nome_municipio'] == municipio) & (df_ml_aluno['ano'] == ano), 'feat_media_portugues_municipio'] = \
          df_ml_aluno.loc[(df_ml_aluno['feat_media_portugues_municipio'].isna() == False) & (df_ml_aluno['nome_municipio'] == municipio) & \
          (df_ml_aluno['ano'] == ano), 'feat_media_portugues_municipio'].unique().mean()

In [184]:
df_ml_aluno[df_ml_aluno['feat_media_portugues_municipio'].isna() == True].head()

,id_aluno,id_escola,nome_municipio,sigla_uf,feat_rede_encoded,feat_peso_aluno,feat_media_proficiencia_escola,feat_media_portugues_municipio,feat_media_portugues_estado,target_alfabetizado,target_nivel_alfabetizacao,_gold_processed_at,ano
